In [325]:
import requests
from langchain_core.prompts import PromptTemplate 
from langchain_core.tools import tool, StructuredTool, InjectedToolArg
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Annotated
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import os
from dotenv import load_dotenv
load_dotenv()

True

In [326]:
NEWS_DATA_API=os.getenv('NEWS_DATA_API')

In [327]:
# # #template to fetch relevant news articles by passing relevant query parameters to the API
# prompt = PromptTemplate(
#     template = """ 
#         You are a financial query builder for the NewsData.io API focusing on the Indian stock market (NSE/BSE).
#         Your goal is to output URL parameters that find single-stock corporate catalysts (earnings, contract wins, mergers, SEBI orders) rather than broad index commentary or holiday announcements.

#         qInTitle:Search news articles for specific keywords or phrases present in the news titles only. The value must be URL-encoded and the maximum character limit permitted is 512 characters.\n
#         qInMeta:Search news articles for specific keywords or phrases present in the news titles, URL, meta keywords and meta description only. The value must be URL-encoded and the maximum character limit permitted is 512 characters.\n
#         q:Search news articles for specific keywords or phrases present in the news title, content, URL, meta keywords and meta description. The value must be URL-encoded and the maximum character limit permitted is 512 characters.\n
#         timeframe:Search the news articles for a specific timeframe (Minutes and Hours). For hours, you can set a timeframe of 1 to 48, and for minutes, you can define a timeframe of 1m to 2880m. For example, if you want to get the news for the past 6 hours then use timeframe=6 and if you want to get news for the last 15 min then use timeframe=15m.\n
#         category:Search news articles for a specific category. The available categories are business, entertainment, environment, food, health, politics, science, sports, technology and top.\n

#         Note - You can only use timeframe either in hours or minutes.
#         Note: qInTitle can't be used with q or qInMeta parameter in the same query.\n
#         Note: qInMeta can't be used with q or qInTitle parameter in the same query.\n

#         1. Target parameter 'q' with actionable corporate catalyst keywords (e.g., "quarterly results" OR "order win" OR "acquisition" OR "SEBI" OR "dividend").
#         2. Do NOT search for broad index phrases like "Sensex", "Nifty", "Indian Stock Market", or "market holiday".
#         3. Always include `category=business`.

#         I need you to generate URL parameters for fetching news articles related to the {topic}.
#  """,

#     input_variables = ["topic"]
# )

In [328]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""You are a query parameter builder for the NewsData.io API focusing strictly on Indian stock market catalysts (NSE/BSE).

Target Topic: {topic}

Step 1: Choose your search scope (CHOOSE EXACTLY ONE):
- 'q=...' (Broad search across the whole article)
- 'qInTitle=...' (Strict search only in the headline)
- 'qInMeta=...' (Search titles, URLs, and meta tags)
- 'timeframe=...' (Search by timeframe)
- 'category=...' (Search by category)

CRITICAL: Never combine these. Pick only one.

Step 2: Apply these mandatory rules:
1. Use actionable single-stock catalyst terms (e.g., "quarterly results", "order win", "acquisition").
2. Strictly DO NOT include broad market phrases like "Sensex" or "Nifty".
3. Always append '&category=business&country=in'.

Validation Rules:
- STRICTLY EXCLUDE foreign equities (e.g., Nike, Tesla, Apple, Google). Only include companies actively listed on the NSE or BSE.
- STRICTLY EXCLUDE general editorial comparisons or educational articles without fresh catalysts.
- For each valid event, state the exact NSE/BSE ticker symbol (e.g., UNIONBANK, INFY, WIPRO).

Output Format:
- Output ONLY the final query string separated by '&'.
- Do NOT include the base URL or conversational explanations.
- Example output: qInTitle=order+win+OR+acquisition&category=business&country=in
""",
    input_variables=["topic"]
)

In [329]:
model = ChatOllama(model="gemma4:e4b", temperature=0.7, num_ctx=8192)

In [330]:
parser = StrOutputParser()

In [331]:
class Article(BaseModel):
    title: str = Field(..., description="The title of the news article"),
    description: str = Field(..., description="A brief description of the news article"),
    source: str = Field(..., description="The source of the news article")

In [332]:
class ArticleList(BaseModel):
    articles: list[Article] = Field(description="List of 5 news articles")

In [333]:
articles_parser = PydanticOutputParser(pydantic_object=ArticleList)


In [334]:
@tool
def generate_query_params(topic: str):
    """Generate a formatted URL query parameter string for fetching news articles."""
    # print("generate_query_params called with topic:", topic)
    model = ChatOllama(model="gemma4:e4b", temperature=0.7, max_tokens=4000)
    chain = prompt | model | parser

    prompt_text = prompt.invoke({'topic': topic})
    print(prompt_text)

    query_params = chain.invoke({'topic' : topic})
    # print(query_params)

    return query_params

In [335]:
class NewsQueryParams(BaseModel):
    topic : str = Field(..., description="The topic to search news articles for")

# @tool
def fetch_news_articles(topic: str):
    """ Tool to fetch news articles. MUST only be called after generating query parameters. """

    query_chain = prompt | model | parser
    query_params = query_chain.invoke({"topic": topic})

    print("query_params_before_fetching: ",query_params)
    
    url = f"https://newsdata.io/api/1/latest?apikey={NEWS_DATA_API}&{query_params}&size=10"
    response = requests.get(url)
    data = response.json()
    
    # 1. Check for API success
    if data.get("status") == "success":
        clean_articles = []
        
        # 2. Extract only the essential text needed by the financial analyst prompt
        for article in data.get("results", []):
            clean_articles.append({
                "title": article.get("title"),
                "description": article.get("description"),
                "source": article.get("source_name")
            })
            
        # 3. Return the minimized list of dictionaries, NOT the raw JSON
        return clean_articles

    return data

fetch_news_articles_tool = StructuredTool.from_function(
    func=fetch_news_articles,
    name="fetch_news_articles",
    description="Fetch relevant news articles by passing relevant query parameters to the API",
    args_schema=NewsQueryParams
)

In [336]:
agent = create_agent(
    model=model,
    tools=[fetch_news_articles_tool],
)

format_instructions = articles_parser.get_format_instructions()

prompt_text = f"""You are an equity research analyst covering individual stocks on the NSE and BSE.

Step 1:
Call `fetch_news_articles` with a query focused on company-specific corporate actions (such as quarterly earnings surprises, mega contract wins, mergers, or regulatory penalties).

Step 2:
From the fetched results, filter out and ignore:
- Broad market index wrap-ups (e.g., Sensex/Nifty up/down commentary).
- Trading holiday calendars and exchange scheduling notices.
- Global macroeconomic recaps (crude oil prices, US bond yields).

Step 3:
Identify 5 specific companies with actionable catalysts capable of moving the individual stock price by 2% or more.

Output strictly according to these formatting instructions:
{format_instructions}"""

response = agent.invoke({"messages": [HumanMessage(content=prompt_text)]})
raw_content = response["messages"][-1].content
# print("Raw content from agent:\n", raw_content)
structured_result = articles_parser.parse(raw_content)

for i, item in enumerate(structured_result.articles):
    print(f"Article {i+1}:\nTitle: {item.title}\nDescription: {item.description}\nSource: {item.source}\n")

c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='A brief description of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


query_params_before_fetching:  qInTitle=quarterly+results+OR+earnings+surprise+OR+order+win+OR+acquisition+OR+merger+OR+regulatory+penalty&category=business&country=in
Article 1:
Title: Infosys Q4 Earnings Beat Expectations on Strong Digital Transformation Demand
Description: Infosys reported a 15% year-on-year revenue jump, significantly surpassing analyst estimates due to robust demand in the AI and cloud segments, presenting a strong upward catalyst for the stock.
Source: Business Standard

Article 2:
Title: Larsen & Toubro Secures Mega Contract for Coastal Defense Project
Description: L&T has won a substantial, multi-year contract from the Ministry of Defence for a major coastal defense system, expected to boost order book visibility and revenue by over 20%.
Source: Economic Times

Article 3:
Title: Adani Ports Announces Strategic Acquisition of Minority Stake in Major Shipping Line
Description: The acquisition move strengthens Adani Ports' logistics footprint in the regional marke